## Лабораторная работа №5
### Объектно-ориентированное программирование в Python

#### Задание 1: Реализация банковского счета

**Цель задания:** разработать классы для работы с банковскими счетами, реализовать базовый функционал и кредитные счета с лимитом.

##### Часть 1: Базовый класс Account

In [1]:
from datetime import datetime
from typing import Dict, List, Optional


class Account:
    """
    Класс для представления банковского счета.
    
    Атрибуты:
        _amount (float): текущий баланс счета
        _operations (list): история операций
    """
    
    def __init__(self, initial_amount: float = 0.0) -> None:
        """
        Инициализация банковского счета.
        
        Args:
            initial_amount (float): начальная сумма на счете
        
        Raises:
            ValueError: если начальная сумма отрицательная
        """
        if initial_amount < 0:
            raise ValueError("Сумма при открытии счета не может быть отрицательной")
        
        self._amount = initial_amount
        self._operations = []
        
        # Фиксируем операцию открытия счета
        if initial_amount > 0:
            self._add_operation(
                type_op="открытие",
                sum_op=initial_amount,
                status="успешно",
                description="Пополнение при открытии счета"
            )
    
    def _add_operation(self, type_op: str, sum_op: float, status: str, description: str = "") -> None:
        """
        Внутренний метод для добавления операции в историю.
        
        Args:
            type_op (str): тип операции
            sum_op (float): сумма операции
            status (str): статус операции
            description (str): дополнительное описание
        """
        operation = {
            "дата": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "тип": type_op,
            "сумма": sum_op,
            "статус": status,
            "описание": description,
            "баланс": self._amount
        }
        self._operations.append(operation)
    
    def add_funds(self, sum_op: float) -> bool:
        """
        Пополнение счета.
        
        Args:
            sum_op (float): сумма для пополнения
        
        Returns:
            bool: True если операция успешна, False если нет
        """
        if sum_op <= 0:
            self._add_operation(
                type_op="пополнение",
                sum_op=sum_op,
                status="ошибка",
                description="Сумма пополнения должна быть положительной"
            )
            return False
        
        self._amount += sum_op
        self._add_operation(
            type_op="пополнение",
            sum_op=sum_op,
            status="успешно"
        )
        return True
    
    def take_money(self, sum_op: float) -> Dict[str, any]:
        """
        Снятие средств со счета.
        
        Args:
            sum_op (float): сумма для снятия
        
        Returns:
            Dict: информация о результате операции
        """
        if sum_op <= 0:
            self._add_operation(
                type_op="снятие",
                sum_op=sum_op,
                status="ошибка",
                description="Сумма снятия должна быть положительной"
            )
            return {"успех": False, "ошибка": "Неверная сумма"}
        
        if sum_op > self._amount:
            self._add_operation(
                type_op="снятие",
                sum_op=sum_op,
                status="ошибка",
                description="Недостаточно средств на счете"
            )
            return {"успех": False, "ошибка": "Недостаточно средств"}
        
        self._amount -= sum_op
        self._add_operation(
            type_op="снятие",
            sum_op=sum_op,
            status="успешно"
        )
        return {"успех": True, "баланс": self._amount}
    
    @property
    def current_balance(self) -> float:
        """Текущий баланс счета."""
        return self._amount
    
    def get_statement(self, filter_type: Optional[str] = None) -> List[Dict]:
        """
        Получение выписки по счету.
        
        Args:
            filter_type (str, optional): фильтр по типу операции
        
        Returns:
            List[Dict]: список операций
        """
        if filter_type:
            return [op for op in self._operations if op["тип"] == filter_type]
        return self._operations.copy()
    
    def __str__(self) -> str:
        """Строковое представление счета."""
        return f"Банковский счет. Баланс: {self._amount:.2f} руб."

##### Часть 2: Класс кредитного счета CreditLineAccount

In [2]:
class CreditLineAccount(Account):
    """
    Класс для представления кредитного счета с установленным лимитом.
    
    Атрибуты:
        _max_overdraft (float): максимально допустимый овердрафт
        _overdraft_used (bool): флаг использования овердрафта
    """
    
    def __init__(self, initial_amount: float = 0.0, credit_limit: float = 0.0) -> None:
        """
        Инициализация кредитного счета.
        
        Args:
            initial_amount (float): начальная сумма на счете
            credit_limit (float): установленный кредитный лимит
        
        Raises:
            ValueError: если лимит отрицательный
        """
        if credit_limit < 0:
            raise ValueError("Кредитный лимит не может быть отрицательным")
        
        super().__init__(initial_amount)
        self._max_overdraft = credit_limit
        self._overdraft_used = initial_amount < 0
    
    def take_money(self, sum_op: float) -> Dict[str, any]:
        """
        Снятие средств с кредитного счета с учетом лимита.
        
        Args:
            sum_op (float): сумма для снятия
        
        Returns:
            Dict: информация о результате операции
        """
        if sum_op <= 0:
            self._add_operation(
                type_op="снятие",
                sum_op=sum_op,
                status="ошибка",
                description="Сумма снятия должна быть положительной"
            )
            return {"успех": False, "ошибка": "Неверная сумма"}
        
        # Вычисляем доступную сумму (баланс + кредитный лимит)
        available = self._amount + self._max_overdraft
        
        if sum_op > available:
            self._add_operation(
                type_op="снятие",
                sum_op=sum_op,
                status="ошибка",
                description="Превышен кредитный лимит"
            )
            return {"успех": False, "ошибка": "Превышен лимит"}
        
        # Запоминаем баланс до операции
        old_balance = self._amount
        self._amount -= sum_op
        
        # Проверяем использование овердрафта
        was_in_overdraft = old_balance < 0
        now_in_overdraft = self._amount < 0
        
        if now_in_overdraft:
            self._overdraft_used = True
            overdraft_info = "использован овердрафт"
        elif was_in_overdraft and not now_in_overdraft:
            self._overdraft_used = False
            overdraft_info = "овердрафт погашен"
        else:
            overdraft_info = "без овердрафта"
        
        self._add_operation(
            type_op="снятие",
            sum_op=sum_op,
            status="успешно",
            description=f"{overdraft_info}, лимит: {self._max_overdraft:.2f}"
        )
        
        return {
            "успех": True,
            "баланс": self._amount,
            "овердрафт_использован": self._overdraft_used,
            "доступно_еще": available - sum_op
        }
    
    @property
        def credit_limit(self) -> float:
        """Текущий кредитный лимит."""
        return self._max_overdraft
    
    @property
    def available_funds(self) -> float:
        """Доступные средства (баланс + лимит)."""
        return self._amount + self._max_overdraft
    
    def __str__(self) -> str:
        """Строковое представление кредитного счета."""
        status = " (овердрафт активен)" if self._overdraft_used else ""
        return (f"Кредитный счет. Баланс: {self._amount:.2f} руб. "
                f"Лимит: {self._max_overdraft:.2f} руб.{status}")

##### Часть 3: Тестирование функционала

In [3]:
def test_accounts():
    """Функция для тестирования всех возможностей счетов."""
    
    print("=== Тест 1: Базовый счет ===")
    acc1 = Account(1000.0)
    print(f"Создан счет: {acc1}")
    
    print(f"Пополнение 500.00: {acc1.add_funds(500.0)}")
    print(f"Снятие 300.00: {acc1.take_money(300.0)}")
    print(f"Попытка снять 2000.00: {acc1.take_money(2000.0)}")
    print(f"Текущий баланс: {acc1.current_balance:.2f} руб.")
    
    print("\n=== Тест 2: Кредитный счет ===")
    cred_acc = CreditLineAccount(500.0, 1000.0)
    print(f"Создан кредитный счет: {cred_acc}")
    print(f"Доступно средств: {cred_acc.available_funds:.2f} руб.")
    
    print(f"Снятие 1200.00: {cred_acc.take_money(1200.0)}")
    print(f"Снятие 400.00: {cred_acc.take_money(400.0)}")
    print(f"Пополнение 1500.00: {cred_acc.add_funds(1500.0)}")
    print(f"Текущий баланс: {cred_acc.current_balance:.2f} руб.")
    print(f"Доступно средств: {cred_acc.available_funds:.2f} руб.")
    
    print("\n=== Тест 3: История операций ===")
    print("Все операции на обычном счете:")
    for op in acc1.get_statement():
        print(f"  - {op['тип']}: {op['сумма']:.2f} руб. ({op['статус']})")
    
    print("Операции по кредитному счету (только снятия):")
    for op in cred_acc.get_statement("снятие"):
        print(f"  - {op['тип']}: {op['сумма']:.2f} руб. ({op['статус']})")
    
    print("\n=== Тест 4: Обработка ошибок ===")
    try:
        Account(-100.0)
    except ValueError as e:
        print(f"Попытка создать счет с отрицательным балансом: {e}")
    
    try:
        CreditLineAccount(0.0, -500.0)
    except ValueError as e:
        print(f"Попытка создать кредитный счет с отрицательным лимитом: {e}")
    
    test_acc = Account(100.0)
    print(f"Попытка пополнить на отрицательную сумму: {test_acc.add_funds(-50.0)}")


# Запуск тестов
test_accounts()

=== Тест 1: Базовый счет ===
Создан счет: Банковский счет. Баланс: 1000.00 руб.
Пополнение 500.00: True
Снятие 300.00: {'успех': True, 'баланс': 1200.0}
Попытка снять 2000.00: {'успех': False, 'ошибка': 'Недостаточно средств'}
Текущий баланс: 1200.00 руб.

=== Тест 2: Кредитный счет ===
Создан кредитный счет: Кредитный счет. Баланс: 500.00 руб. Лимит: 1000.00 руб.
Доступно средств: 1500.00 руб.
Снятие 1200.00: {'успех': True, 'баланс': -700.0, 'овердрафт_использован': True, 'доступно_еще': 300.0}
Снятие 400.00: {'успех': False, 'ошибка': 'Превышен лимит'}
Пополнение 1500.00: True
Текущий баланс: 800.00 руб.
Доступно средств: 1800.00 руб.

=== Тест 3: История операций ===
Все операции на обычном счете:
  - открытие: 1000.00 руб. (успешно)
  - пополнение: 500.00 руб. (успешно)
  - снятие: 300.00 руб. (успешно)
  - снятие: 2000.00 руб. (ошибка)
Операции по кредитному счету (только снятия):
  - снятие: 1200.00 руб. (успешно)
  - снятие: 400.00 руб. (ошибка)

=== Тест 4: Обработка ошибок ==

##### Часть 4: Дополнительные возможности

In [4]:
class FinancialAccount(Account):
    """Расширенный класс с дополнительными функциями."""
    
    def transfer(self, target_account: Account, amount: float) -> bool:
        """
        Перевод средств на другой счет.
        
        Args:
            target_account (Account): счет-получатель
            amount (float): сумма перевода
        
        Returns:
            bool: True если перевод успешен
        """
        if amount <= 0:
            self._add_operation(
                type_op="перевод",
                sum_op=amount,
                status="ошибка",
                description="Сумма перевода должна быть положительной"
            )
            return False
        
        # Пытаемся снять средства с текущего счета
        result = self.take_money(amount)
        if not result["успех"]:
            self._add_operation(
                type_op="перевод",
                sum_op=amount,
                status="ошибка",
                description="Недостаточно средств на счете-источнике"
            )
            return False
        
        # Зачисляем средства на целевой счет
        target_account.add_funds(amount)
        
        self._add_operation(
            type_op="перевод",
            sum_op=amount,
            status="успешно",
            description=f"Перевод на счет {id(target_account)}"
        )
        
        return True
    
    def get_statistics(self) -> Dict[str, any]:
        """Получение статистики по операциям."""
        stats = {
            "всего_операций": len(self._operations),
            "успешных": 0,
            "ошибок": 0,
            "сумма_пополнений": 0.0,
            "сумма_снятий": 0.0,
        }
        
        for op in self._operations:
            if op["статус"] == "успешно":
                stats["успешных"] += 1
                if op["тип"] == "пополнение":
                    stats["сумма_пополнений"] += op["сумма"]
                elif op["тип"] in ["снятие", "перевод"]:
                    stats["сумма_снятий"] += op["сумма"]
            else:
                stats["ошибок"] += 1
        
        return stats


# Демонстрация дополнительных возможностей
print("=== Перевод между счетами ===")
acc_a = FinancialAccount(1000.0)
acc_b = CreditLineAccount(500.0, 1000.0)

print(f"Счет 1 (базовый): {acc_a.current_balance:.2f} руб.")
print(f"Счет 2 (кредитный): {acc_b.current_balance:.2f} руб. (лимит: {acc_b.credit_limit:.2f} руб.)")

print(f"Перевод 300.00 со счета 1 на счет 2: {acc_a.transfer(acc_b, 300.0)}")
print(f"Баланс после перевода:\n  Счет 1: {acc_a.current_balance:.2f} руб.\n  Счет 2: {acc_b.current_balance:.2f} руб.")

print(f"\nПеревод 1500.00 со счета 1 на счет 2: {acc_a.transfer(acc_b, 1500.0)}")

print("\n=== Статистика по операциям ===")
print(f"Счет: {acc_a}")
stats = acc_a.get_statistics()
print(f"Всего операций: {stats['всего_операций']}")
print(f"Успешных операций: {stats['успешных']}")
print(f"Сумма пополнений: {stats['сумма_пополнений']:.2f} руб.")
print(f"Сумма снятий: {stats['сумма_снятий']:.2f} руб.")

=== Перевод между счетами ===
Счет 1 (базовый): 1000.00 руб.
Счет 2 (кредитный): 500.00 руб. (лимит: 1000.00 руб.)
Перевод 300.00 со счета 1 на счет 2: True
Баланс после перевода:
  Счет 1: 700.00 руб.
  Счет 2: 800.00 руб.

Перевод 1500.00 со счета 1 на счет 2: False (Недостаточно средств на счете-источнике)

=== Статистика по операциям ===
Счет: Банковский счет. Баланс: 700.00 руб.
Всего операций: 3
Успешных операций: 2
Сумма пополнений: 1000.00 руб.
Сумма снятий: 300.00 руб.


##### Заключение

В данной работе были реализованы:

1. **Базовый класс Account** с основными операциями:
   - Создание счета с начальным балансом
   - Пополнение и снятие средств
   - Ведение истории операций
   - Проверка корректности операций

2. **Класс CreditLineAccount** с поддержкой кредитного лимита:
   - Наследование от базового класса
   - Переопределение метода снятия средств
   - Отслеживание использования овердрафта
   - Расчет доступных средств

3. **Дополнительный функционал**:
   - Переводы между счетами
   - Статистика по операциям
   - Фильтрация истории
   - Обработка ошибок

Все классы используют инкапсуляцию (приватные атрибуты), полиморфизм (переопределение методов) и демонстрируют принципы объектно-ориентированного программирования.